# 02 — Feature Engineering

## KKBOX Customer Churn Prediction Platform

### Objective

Notebook này kiểm tra kết quả của **QT3 — Data Processing & Feature Engineering**.

Production feature pipeline đã được triển khai trong:

```text
src/kkbox_churn_prediction/
├── data/
│   └── validation.py
└── features/
    ├── members.py
    ├── transactions.py
    ├── user_logs.py
    └── build_dataset.py


---

# CELL 2 — Markdown

```markdown
## 1. Setup

In [33]:
import duckdb
import pandas as pd

from kkbox_churn_prediction.config import (
    CUTOFF_DATE,
    MEMBER_FEATURES_PATH,
    TRANSACTION_FEATURES_PATH,
    USER_LOG_FEATURES_PATH,
    TRAIN_FEATURES_PATH,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.3f}".format)

print("Cutoff date:", CUTOFF_DATE)
print("Dataset:", TRAIN_FEATURES_PATH)

Cutoff date: 2017-04-01
Dataset: /home/namdp/Documents/AI_Projects/kkbox-churn-prediction/data/processed/train_features.parquet


## 2. Load Processed Feature Dataset

Dataset cuối của QT3:

```text
data/processed/train_features.parquet


---

# CELL 5 — Code

Không cần load cả gần 1 triệu rows vào Pandas ngay.

```python
feature_sample = duckdb.sql(
    f"""
    SELECT *
    FROM read_parquet(
        '{TRAIN_FEATURES_PATH.as_posix()}'
    )
    LIMIT 10
    """
).df()

feature_sample

In [34]:
schema = duckdb.sql(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        '{TRAIN_FEATURES_PATH.as_posix()}'
    )
    """
).df()

schema

,column_name,column_type,null,key,default,extra
0,msno,VARCHAR,YES,None,None,None
1,is_churn,BIGINT,YES,None,None,None
2,has_member_data,INTEGER,YES,None,None,None
3,has_transaction_data,INTEGER,YES,None,None,None
4,has_log_data,INTEGER,YES,None,None,None
5,city,BIGINT,YES,None,None,None
6,age,BIGINT,YES,None,None,None
7,gender,VARCHAR,YES,None,None,None
8,registered_via,BIGINT,YES,None,None,None
9,account_age_days,BIGINT,YES,None,None,None


## 3. Dataset Integrity

Kiểm tra các invariant quan trọng:

- Final row count phải bằng `train_v2`.
- `msno` phải unique.
- Target không được missing.
- Target chỉ chứa `0` và `1`.

In [35]:
integrity = duckdb.sql(
    f"""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT msno) AS users,

        SUM(
            CASE WHEN msno IS NULL
            THEN 1 ELSE 0 END
        ) AS missing_msno,

        SUM(
            CASE WHEN is_churn IS NULL
            THEN 1 ELSE 0 END
        ) AS missing_target,

        SUM(
            CASE WHEN is_churn NOT IN (0, 1)
            THEN 1 ELSE 0 END
        ) AS invalid_target

    FROM read_parquet(
        '{TRAIN_FEATURES_PATH.as_posix()}'
    )
    """
).df()

integrity

,rows,users,missing_msno,missing_target,invalid_target
0,970960,970960,0.000,0.000,0.000


### Findings

- Final dataset có **970,960 rows**.
- Có **970,960 unique users**.
- `msno` không missing.
- `is_churn` không missing.
- Target chỉ chứa hai class hợp lệ `0` và `1`.
- Dataset đáp ứng invariant quan trọng:

> **1 msno = 1 row**

## 4. Source Coverage

Do sử dụng `LEFT JOIN`, không phải tất cả user đều có dữ liệu trong `members`, `transactions` và `user_logs`.

Các feature:

- `has_member_data`
- `has_transaction_data`
- `has_log_data`

được sử dụng để ghi nhận trạng thái này.

In [36]:
coverage = duckdb.sql(
    f"""
    SELECT
        AVG(has_member_data) AS member_coverage,
        AVG(has_transaction_data) AS transaction_coverage,
        AVG(has_log_data) AS log_coverage

    FROM read_parquet(
        '{TRAIN_FEATURES_PATH.as_posix()}'
    )
    """
).df()

coverage

,member_coverage,transaction_coverage,log_coverage
0,0.887,0.961,0.777


### Coverage Findings

- Member coverage: khoảng **88.67%**.
- Transaction coverage: khoảng **96.15%**.
- User log coverage: khoảng **77.71%**.

Các kết quả này khớp với Data Understanding ở QT2.

Điều đó xác nhận quá trình aggregation và LEFT JOIN không làm thay đổi đáng kể coverage của các source table.

## 5. Member Features

Member feature set gồm:

- `city`
- `age`
- `gender`
- `registered_via`
- `account_age_days`
- `has_member_data`

In [37]:
member_features = duckdb.sql(
    f"""
    SELECT
        msno,
        has_member_data,
        city,
        age,
        gender,
        registered_via,
        account_age_days

    FROM read_parquet(
        '{TRAIN_FEATURES_PATH.as_posix()}'
    )

    LIMIT 20
    """
).df()

member_features

,msno,has_member_data,city,age,gender,registered_via,account_age_days
0,FhPCgS2PwrnNot9ZcA6N8pWZ0oWrKvJkcKBEylQ9Gdw=,1,13,37,female,9,3410
1,468MQnwdDufMzyb9vP+hTDhSAtXX0lPqpgh0Aq7mw+k=,1,13,32,female,9,3400
2,5X1OUve67UaVC7YNRqEjsNIDfujkI9DGLs88saplE7s=,1,5,22,female,9,3398
3,749fJcKXBgC8vpWoDnq1ey50s0rrdfcOFssdxZNyxQ0=,1,13,23,female,9,3398
4,MNRUD2pAtKpbaPsD2bJqhwKQsIt06ZkosKVWXFZI2TQ=,1,22,29,female,9,3395
5,GH3m4Nd0GJln0mXZDe6/5HJnSpYWOpEAWi6tANMjwSY=,1,13,53,female,9,3395
6,P4iI9a7POI3XXCu5CtNY4RWWMICD8WmvIv7EInMEa6Y=,1,13,28,male,9,3394
7,Q2kK1TETqeMB5zzptjaEdlRvWRpGUBSjc8GacCIYOLI=,1,5,33,female,9,3391
8,8fj9nyhQwB8e4EEIbmA5CNHQ4ZgMDy4JhuLNgrngY+w=,1,5,30,female,9,3383
9,EKqCXPsMTmCvJ6YQS7TRJJ4KcfDYoHUB+6VPdmTAgEc=,1,10,50,female,9,3373


In [38]:
age_summary = duckdb.sql(
    f"""
    SELECT
        COUNT(*) AS total_users,

        SUM(
            CASE WHEN age IS NULL
            THEN 1 ELSE 0 END
        ) AS missing_age,

        MIN(age) AS min_age,
        MEDIAN(age) AS median_age,
        AVG(age) AS mean_age,
        MAX(age) AS max_age

    FROM read_parquet(
        '{TRAIN_FEATURES_PATH.as_posix()}'
    )
    """
).df()

age_summary

,total_users,missing_age,min_age,median_age,mean_age,max_age
0,970960,"584,245.000",1,28.000,29.904,100


In [39]:
duckdb.sql(
    f"""
    SELECT COUNT(*) AS invalid_age_after_cleaning

    FROM read_parquet(
        '{TRAIN_FEATURES_PATH.as_posix()}'
    )

    WHERE age IS NOT NULL
      AND (
          age < 1
          OR age > 100
      )
    """
).df()

,invalid_age_after_cleaning
0,0


In [40]:
duckdb.sql(
    f"""
    SELECT
        gender,
        COUNT(*) AS users,
        ROUND(
            COUNT(*) * 100.0
            / SUM(COUNT(*)) OVER (),
            2
        ) AS percentage

    FROM read_parquet(
        '{TRAIN_FEATURES_PATH.as_posix()}'
    )

    GROUP BY gender
    ORDER BY users DESC
    """
).df()

,gender,users,percentage
0,unknown,472062,48.620
1,male,204561,21.070
2,female,184344,18.990
3,NaN,109993,11.330


In [41]:
duckdb.sql(
    f"""
    SELECT
        MIN(account_age_days) AS min_days,
        MEDIAN(account_age_days) AS median_days,
        AVG(account_age_days) AS mean_days,
        MAX(account_age_days) AS max_days,

        SUM(
            CASE
                WHEN account_age_days < 0
                THEN 1 ELSE 0
            END
        ) AS negative_account_age

    FROM read_parquet(
        '{TRAIN_FEATURES_PATH.as_posix()}'
    )
    """
).df()

,min_days,median_days,mean_days,max_days,negative_account_age
0,1,"1,034.000","1,295.703",4754,0.000


### Member Feature Findings

- Invalid `bd` values đã được chuyển thành missing thay vì drop user.
- Không còn age ngoài miền `1–100`.
- Missing gender được biểu diễn bằng category `unknown` khi member record tồn tại.
- User không có member profile được nhận diện qua `has_member_data`.
- `account_age_days` được tính tương đối với `CUTOFF_DATE`.

## 6. Transaction Features

Transaction records đã được aggregate:

```text
N transaction rows
        ↓
GROUP BY msno
        ↓
1 row / user


---

# CELL 21 — Code: Intermediate integrity

Rất nên kiểm tra từng interim file.

```python
transaction_integrity = duckdb.sql(
    f"""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT msno) AS users

    FROM read_parquet(
        '{TRANSACTION_FEATURES_PATH.as_posix()}'
    )
    """
).df()

transaction_integrity

In [42]:
duckdb.sql(
    f"""
    SELECT *
    FROM read_parquet(
        '{TRANSACTION_FEATURES_PATH.as_posix()}'
    )
    LIMIT 10
    """
).df()

,msno,transaction_count,total_paid,avg_paid,min_paid,max_paid,avg_plan_price,avg_plan_days,avg_discount,auto_renew_rate,last_auto_renew,cancel_count,cancel_rate,last_is_cancel,last_transaction_date,days_since_last_transaction
0,5hxT0qvQO2nhomxhH0nWXLmu5fJZjmQC3a5nYlPZNLQ=,2,198.000,99.000,99,99,99.000,30.000,0.000,1.000,1,0.000,0.000,0,2017-03-31,1
1,64aWgdQmoW5cquEAPlMavsjvipwDvZjks0wo/V3hl7I=,1,149.000,149.000,149,149,149.000,30.000,0.000,1.000,1,0.000,0.000,0,2017-03-04,28
2,6OBgbQa/EcagK/oj4YvflByqP1xo3Gkjz5xGSLQL+1g=,1,180.000,180.000,180,180,180.000,30.000,0.000,1.000,1,0.000,0.000,0,2017-03-06,26
3,6bndnutu2QKB3uiRE/CFAyNucALMc3gNaVlQOmbH6EU=,2,198.000,99.000,99,99,99.000,30.000,0.000,1.000,1,1.000,0.500,1,2017-03-13,19
4,7DcaMY3qdkBw98qtszWbqQ7/R+oVIJSVSAPcN0fpF18=,1,99.000,99.000,99,99,99.000,30.000,0.000,1.000,1,0.000,0.000,0,2017-03-06,26
5,7UZeWybuq4azuyoqhLeAplpx+xyk4IHXco08nOm7tE0=,3,447.000,149.000,149,149,149.000,30.000,0.000,1.000,1,0.000,0.000,0,2017-03-31,1
6,7aSUJSyHlsM5SP51cg/kylODL6Ek7DglUTHcbsCcPis=,1,99.000,99.000,99,99,99.000,30.000,0.000,1.000,1,0.000,0.000,0,2017-03-31,1
7,8NUjdocOoyz2HTCI3YStcsgn6PC7bqBa2ezWxnz/0xk=,2,298.000,149.000,149,149,149.000,30.000,0.000,1.000,1,0.000,0.000,0,2017-03-31,1
8,AEX8ker9jO8pWs27NMJHySMetP9Ru5DIvgZiUyWrP1M=,1,99.000,99.000,99,99,99.000,30.000,0.000,1.000,1,0.000,0.000,0,2017-03-01,31
9,ATVhq5EQJsHMMle7vhENLNeIkMeHH8ZhP88nkKJg2fk=,1,149.000,149.000,149,149,149.000,30.000,0.000,1.000,1,0.000,0.000,0,2017-03-23,9


In [43]:
duckdb.sql(
    f"""
    SELECT
        MIN(transaction_count) AS min_tx,
        MEDIAN(transaction_count) AS median_tx,
        AVG(transaction_count) AS mean_tx,
        MAX(transaction_count) AS max_tx,

        MEDIAN(avg_paid) AS median_avg_paid,
        AVG(avg_paid) AS mean_avg_paid,

        AVG(auto_renew_rate) AS mean_auto_renew_rate,
        AVG(cancel_rate) AS mean_cancel_rate,

        MIN(days_since_last_transaction)
            AS min_transaction_recency,

        MEDIAN(days_since_last_transaction)
            AS median_transaction_recency,

        MAX(days_since_last_transaction)
            AS max_transaction_recency

    FROM read_parquet(
        '{TRANSACTION_FEATURES_PATH.as_posix()}'
    )
    """
).df()

,min_tx,median_tx,mean_tx,max_tx,median_avg_paid,mean_avg_paid,mean_auto_renew_rate,mean_cancel_rate,min_transaction_recency,median_transaction_recency,max_transaction_recency
0,1,1.000,1.195,208,149.000,299.201,0.770,0.017,1,17.000,820


In [44]:
duckdb.sql(
    f"""
    SELECT
        SUM(
            CASE
                WHEN auto_renew_rate < 0
                  OR auto_renew_rate > 1
                THEN 1 ELSE 0
            END
        ) AS invalid_auto_renew_rate,

        SUM(
            CASE
                WHEN cancel_rate < 0
                  OR cancel_rate > 1
                THEN 1 ELSE 0
            END
        ) AS invalid_cancel_rate

    FROM read_parquet(
        '{TRANSACTION_FEATURES_PATH.as_posix()}'
    )
    """
).df()

,invalid_auto_renew_rate,invalid_cancel_rate
0,0.000,0.000


### Transaction Feature Findings

- Transaction history đã được aggregate thành 1 row/user.
- `auto_renew_rate` và `cancel_rate` nằm trong miền `[0, 1]`.
- `transaction_count` biểu diễn frequency.
- `days_since_last_transaction` biểu diễn recency.
- Các feature payment được giữ ở dạng numeric và chưa scaling.
- Không sử dụng `membership_expire_date` trong feature V1 do QT2 phát hiện các giá trị expiration rất xa trong tương lai.

## 7. User Log Features

Daily listening logs đã được aggregate thành user-level behavior features.

Feature groups:

- Activity frequency.
- Listening duration.
- Song completion behavior.
- Unique songs.
- Recency.
- Recent 7-day activity.
- Recent 30-day activity.
- Extreme listening indicators.

In [45]:
logs_integrity = duckdb.sql(
    f"""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT msno) AS users

    FROM read_parquet(
        '{USER_LOG_FEATURES_PATH.as_posix()}'
    )
    """
).df()

logs_integrity

,rows,users
0,1103894,1103894


In [46]:
duckdb.sql(
    f"""
    SELECT *
    FROM read_parquet(
        '{USER_LOG_FEATURES_PATH.as_posix()}'
    )
    LIMIT 10
    """
).df()

,msno,active_days,total_listening_secs,avg_listening_secs_per_day,median_listening_secs_per_day,avg_unique_songs,total_unique_song_events,total_num_25,total_num_50,total_num_75,total_num_985,total_num_100,completion_rate,last_active_date,days_since_last_activity,active_days_7d,active_days_30d,listening_secs_7d,listening_secs_30d,extreme_listening_day_count
0,t32tFkIZm2ZQyybntPXmgMMU/58X39rwvA8MO10yS4Y=,30,"229,224.233","7,640.808","5,321.609",34.300,"1,029.000",190.000,67.000,46.000,40.000,867.000,0.717,2017-03-31,1,7.000,29.000,"58,756.015","228,538.782",0.000
1,+b8TD9xoXzoKhLcDFmhz1itNSXH8lolb6em2jsCLdb8=,9,"32,806.222","3,645.136","1,122.636",15.778,142.000,18.000,10.000,5.000,5.000,116.000,0.753,2017-03-29,3,2.000,9.000,"25,071.555","32,806.222",0.000
2,E097qQj1MDx80k9ChjD5FcwwVr2LCzqsm4zRAx3srkk=,23,"287,110.007","12,483.044","7,793.127",30.348,698.000,40.000,13.000,8.000,4.000,"1,105.000",0.944,2017-03-29,3,4.000,23.000,"25,324.374","287,110.007",0.000
3,Ghakf2zvC00ALJmDY2IyFA+5+KPqOTOJFbxobLxPsJc=,12,"13,124.321","1,093.693",967.832,3.583,43.000,7.000,4.000,5.000,4.000,47.000,0.701,2017-03-28,4,2.000,12.000,"2,108.163","13,124.321",0.000
4,y7WsBYkqb1+sPHvaNiehodqtIcjUWhQmk0pFVka859U=,9,"39,289.067","4,365.452","2,138.372",10.556,95.000,37.000,13.000,6.000,1.000,120.000,0.678,2017-03-28,4,2.000,9.000,"8,852.906","39,289.067",0.000
5,bCJfo1BzmLE0c+SYVYIq3uqtbtRpvsj4/yEnGLrpMcI=,27,"183,224.442","6,786.090","6,096.547",27.556,744.000,112.000,34.000,25.000,17.000,680.000,0.783,2017-03-31,1,5.000,26.000,"39,705.614","168,655.982",0.000
6,4yE2+gIMy3rvW4oclF6vD8HsbauB4EwgPpBWKuSpBuM=,31,"142,628.618","4,600.923","3,293.511",32.935,"1,021.000",505.000,145.000,53.000,50.000,442.000,0.370,2017-03-31,1,7.000,30.000,"23,277.991","139,592.986",0.000
7,jr8G7hc2rvmpXe7MozUnuWriVYKOJEr2J2IdEXT5YCc=,25,"95,838.789","3,833.552","3,118.072",20.200,505.000,252.000,16.000,9.000,15.000,382.000,0.567,2017-03-31,1,7.000,24.000,"14,612.761","88,064.548",0.000
8,VQEQW9yKJOMrrINXmt0sRnU1xenw+k77r5dZAqg4Qpk=,30,"164,226.912","5,474.230","4,786.990",21.300,639.000,302.000,51.000,45.000,27.000,639.000,0.601,2017-03-31,1,6.000,29.000,"27,467.299","158,357.902",0.000
9,C6fJFJ7DaHo7dhm+6+XihJ3Vz+MsnMDstVea9YK5QIM=,30,"557,296.095","18,576.536","14,779.761",9.700,291.000,51.000,14.000,4.000,4.000,"1,332.000",0.948,2017-03-30,2,6.000,29.000,"140,064.780","538,217.285",0.000


In [47]:
duckdb.sql(
    f"""
    SELECT
        MIN(active_days) AS min_active_days,
        MEDIAN(active_days) AS median_active_days,
        AVG(active_days) AS mean_active_days,
        MAX(active_days) AS max_active_days,

        MEDIAN(total_listening_secs)
            AS median_total_secs,

        AVG(total_listening_secs)
            AS mean_total_secs,

        MEDIAN(days_since_last_activity)
            AS median_activity_recency,

        AVG(active_days_7d)
            AS mean_active_days_7d,

        AVG(active_days_30d)
            AS mean_active_days_30d

    FROM read_parquet(
        '{USER_LOG_FEATURES_PATH.as_posix()}'
    )
    """
).df()

,min_active_days,median_active_days,mean_active_days,max_active_days,median_total_secs,mean_total_secs,median_activity_recency,mean_active_days_7d,mean_active_days_30d
0,1,18.000,16.665,31,"73,828.011","131,559.486",1.000,3.770,16.143


In [48]:
duckdb.sql(
    f"""
    SELECT
        SUM(
            CASE
                WHEN active_days_7d > 7
                THEN 1 ELSE 0
            END
        ) AS invalid_active_days_7d,

        SUM(
            CASE
                WHEN active_days_30d > 30
                THEN 1 ELSE 0
            END
        ) AS invalid_active_days_30d,

        SUM(
            CASE
                WHEN active_days_7d > active_days_30d
                THEN 1 ELSE 0
            END
        ) AS invalid_window_order

    FROM read_parquet(
        '{USER_LOG_FEATURES_PATH.as_posix()}'
    )
    """
).df()

,invalid_active_days_7d,invalid_active_days_30d,invalid_window_order
0,0.000,0.000,0.000


In [49]:
duckdb.sql(
    f"""
    SELECT
        SUM(
            CASE
                WHEN listening_secs_7d
                     > listening_secs_30d
                THEN 1 ELSE 0
            END
        ) AS invalid_listening_window

    FROM read_parquet(
        '{USER_LOG_FEATURES_PATH.as_posix()}'
    )
    """
).df()

,invalid_listening_window
0,0.000


In [50]:
duckdb.sql(
    f"""
    SELECT
        MIN(completion_rate) AS min_completion_rate,
        MEDIAN(completion_rate) AS median_completion_rate,
        AVG(completion_rate) AS mean_completion_rate,
        MAX(completion_rate) AS max_completion_rate,

        SUM(
            CASE
                WHEN completion_rate < 0
                  OR completion_rate > 1
                THEN 1 ELSE 0
            END
        ) AS invalid_completion_rate

    FROM read_parquet(
        '{USER_LOG_FEATURES_PATH.as_posix()}'
    )
    """
).df()

,min_completion_rate,median_completion_rate,mean_completion_rate,max_completion_rate,invalid_completion_rate
0,0.000,0.731,0.679,1.000,0.000


In [51]:
duckdb.sql(
    f"""
    SELECT
        COUNT(*) AS users,

        SUM(
            CASE
                WHEN extreme_listening_day_count > 0
                THEN 1 ELSE 0
            END
        ) AS users_with_extreme_days,

        MAX(extreme_listening_day_count)
            AS max_extreme_days

    FROM read_parquet(
        '{USER_LOG_FEATURES_PATH.as_posix()}'
    )
    """
).df()

,users,users_with_extreme_days,max_extreme_days
0,1103894,"2,638.000",31.000


### User Log Feature Findings

- User logs đã được aggregate về 1 row/user.
- Listening duration bất thường không làm thay đổi raw data; chúng được xử lý trong feature pipeline.
- `extreme_listening_day_count` giữ lại thông tin rằng user từng có activity record bất thường.
- `days_since_last_activity` biểu diễn recency.
- Feature windows 7 ngày và 30 ngày được tạo tương đối với cutoff.
- `active_days_7d <= active_days_30d`.
- `listening_secs_7d <= listening_secs_30d`.

## 8. Missing Values After Feature Engineering

Missing values sau Feature Engineering không phải lúc nào cũng là lỗi.

Có ba nguyên nhân chính:

1. User không tồn tại trong source table.
2. Raw value không hợp lệ và đã được chuyển thành missing.
3. Aggregated statistic không tồn tại vì user không có activity tương ứng.

Không thực hiện global imputation ở QT3.

In [52]:
columns = duckdb.sql(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        '{TRAIN_FEATURES_PATH.as_posix()}'
    )
    """
).df()["column_name"].tolist()

missing_parts = []

for column in columns:
    missing_parts.append(
        f"""
        SELECT
            '{column}' AS feature,
            SUM(
                CASE WHEN "{column}" IS NULL
                THEN 1 ELSE 0 END
            ) AS missing_count,
            ROUND(
                100.0 *
                SUM(
                    CASE WHEN "{column}" IS NULL
                    THEN 1 ELSE 0 END
                ) / COUNT(*),
                2
            ) AS missing_pct

        FROM read_parquet(
            '{TRAIN_FEATURES_PATH.as_posix()}'
        )
        """
    )

missing_query = "\nUNION ALL\n".join(missing_parts)

missing_summary = duckdb.sql(
    f"""
    SELECT *
    FROM (
        {missing_query}
    )
    ORDER BY missing_pct DESC
    """
).df()

missing_summary

,feature,missing_count,missing_pct
0,age,"584,245.000",60.170
1,avg_unique_songs,"216,409.000",22.290
2,completion_rate,"216,409.000",22.290
3,days_since_last_activity,"216,409.000",22.290
4,avg_listening_secs_per_day,"216,409.000",22.290
5,median_listening_secs_per_day,"216,409.000",22.290
6,city,"109,993.000",11.330
7,gender,"109,993.000",11.330
8,registered_via,"109,993.000",11.330
9,account_age_days,"109,994.000",11.330


### Initial Missing Findings

Từ validation hiện tại:

- Missing `age`: **584,245 users**.
- Missing `avg_paid`: **37,382 users**.
- Missing `avg_unique_songs`: **216,409 users**.

Các missing này không được fill toàn cục ở QT3.

Ví dụ:

- User không có transaction → `avg_paid` không tồn tại.
- User không có log → `avg_unique_songs` không tồn tại.
- User có invalid raw age hoặc không có member profile → `age` missing.

Numeric/categorical imputation sẽ được fit trên training split ở QT4 để tránh leakage.

## 9. Feature Distributions

Kiểm tra distribution để phát hiện feature có range hoặc outlier bất thường.

Không thực hiện scaling hoặc model-specific transformation ở QT3.

In [53]:
numeric_summary = duckdb.sql(
    f"""
    SELECT
        MEDIAN(age) AS age_median,

        MEDIAN(account_age_days)
            AS account_age_median,

        MEDIAN(transaction_count)
            AS transaction_count_median,

        MEDIAN(avg_paid)
            AS avg_paid_median,

        MEDIAN(cancel_rate)
            AS cancel_rate_median,

        MEDIAN(active_days)
            AS active_days_median,

        MEDIAN(total_listening_secs)
            AS listening_secs_median,

        MEDIAN(days_since_last_activity)
            AS activity_recency_median

    FROM read_parquet(
        '{TRAIN_FEATURES_PATH.as_posix()}'
    )
    """
).df()

numeric_summary

,age_median,account_age_median,transaction_count_median,avg_paid_median,cancel_rate_median,active_days_median,listening_secs_median,activity_recency_median
0,28.000,"1,034.000",1.000,149.000,0.000,14.000,"49,982.129",1.000


In [54]:
duckdb.sql(
    f"""
    SELECT
        QUANTILE_CONT(
            total_paid,
            0.50
        ) AS total_paid_p50,

        QUANTILE_CONT(
            total_paid,
            0.95
        ) AS total_paid_p95,

        QUANTILE_CONT(
            total_paid,
            0.99
        ) AS total_paid_p99,

        QUANTILE_CONT(
            total_listening_secs,
            0.50
        ) AS listening_p50,

        QUANTILE_CONT(
            total_listening_secs,
            0.95
        ) AS listening_p95,

        QUANTILE_CONT(
            total_listening_secs,
            0.99
        ) AS listening_p99

    FROM read_parquet(
        '{TRAIN_FEATURES_PATH.as_posix()}'
    )
    """
).df()

,total_paid_p50,total_paid_p95,total_paid_p99,listening_p50,listening_p95,listening_p99
0,149.000,298.000,"1,299.000","49,982.129","412,943.803","848,807.756"


## 10. Leakage Validation

`CUTOFF_DATE = 2017-04-01`

Không được sử dụng:

```text
transaction_date >= 2017-04-01
activity_date >= 2017-04-01

In [55]:
transaction_leakage = duckdb.sql(
    f"""
    SELECT
        MAX(last_transaction_date)
            AS max_transaction_date,

        SUM(
            CASE
                WHEN last_transaction_date
                     >= DATE '{CUTOFF_DATE.isoformat()}'
                THEN 1 ELSE 0
            END
        ) AS invalid_users

    FROM read_parquet(
        '{TRANSACTION_FEATURES_PATH.as_posix()}'
    )
    """
).df()

transaction_leakage

,max_transaction_date,invalid_users
0,2017-03-31,0.000


In [56]:
log_leakage = duckdb.sql(
    f"""
    SELECT
        MAX(last_active_date)
            AS max_activity_date,

        SUM(
            CASE
                WHEN last_active_date
                     >= DATE '{CUTOFF_DATE.isoformat()}'
                THEN 1 ELSE 0
            END
        ) AS invalid_users

    FROM read_parquet(
        '{USER_LOG_FEATURES_PATH.as_posix()}'
    )
    """
).df()

log_leakage

,max_activity_date,invalid_users
0,2017-03-31,0.000


### Leakage Findings

- `CUTOFF_DATE`: **2017-04-01**.
- Latest transaction used: **2017-03-31**.
- Latest listening activity used: **2017-03-31**.
- Không phát hiện transaction hoặc listening activity xảy ra tại hoặc sau cutoff.

Feature pipeline hiện đáp ứng time-based leakage constraint.

## 11. Target-based Sanity Check

Phần này chỉ dùng để kiểm tra feature có behavior hợp lý giữa churn và non-churn.

Không sử dụng kết quả của section này để tự động lựa chọn feature trước Train/Validation/Test split.

In [57]:
churn_feature_summary = duckdb.sql(
    f"""
    SELECT
        is_churn,

        COUNT(*) AS users,

        AVG(transaction_count)
            AS avg_transaction_count,

        AVG(auto_renew_rate)
            AS avg_auto_renew_rate,

        AVG(cancel_rate)
            AS avg_cancel_rate,

        AVG(active_days)
            AS avg_active_days,

        MEDIAN(total_listening_secs)
            AS median_listening_secs,

        AVG(days_since_last_activity)
            AS avg_activity_recency

    FROM read_parquet(
        '{TRAIN_FEATURES_PATH.as_posix()}'
    )

    GROUP BY is_churn
    ORDER BY is_churn
    """
).df()

churn_feature_summary

,is_churn,users,avg_transaction_count,avg_auto_renew_rate,avg_cancel_rate,avg_active_days,median_listening_secs,avg_activity_recency
0,0,883630,1.153,0.935,0.006,14.097,"51,167.386",3.291
1,1,87330,1.301,0.568,0.243,12.327,"38,419.620",7.919


### Sanity Check Interpretation

QT2 đã quan sát:

- Churn users có auto-renew thấp hơn.
- Churn users có cancellation cao hơn.
- Churn users có ít active days hơn.

Feature dataset sau QT3 nên tiếp tục phản ánh các pattern cơ bản này.

Nếu các pattern biến mất hoàn toàn, cần kiểm tra lại aggregation hoặc cleaning pipeline.

Các quan sát này không được xem là bằng chứng nhân quả.

## 12. Final Feature Set

In [58]:
feature_columns = [
    column
    for column in columns
    if column not in {
        "msno",
        "is_churn",
    }
]

print("Total model candidate features:", len(feature_columns))

for feature in feature_columns:
    print(feature)

Total model candidate features: 40
has_member_data
has_transaction_data
has_log_data
city
age
gender
registered_via
account_age_days
transaction_count
total_paid
avg_paid
min_paid
max_paid
avg_plan_price
avg_plan_days
avg_discount
auto_renew_rate
last_auto_renew
cancel_count
cancel_rate
last_is_cancel
days_since_last_transaction
active_days
total_listening_secs
avg_listening_secs_per_day
median_listening_secs_per_day
avg_unique_songs
total_unique_song_events
total_num_25
total_num_50
total_num_75
total_num_985
total_num_100
completion_rate
days_since_last_activity
active_days_7d
active_days_30d
listening_secs_7d
listening_secs_30d
extreme_listening_day_count


In [59]:
member_feature_names = [
    "has_member_data",
    "city",
    "age",
    "gender",
    "registered_via",
    "account_age_days",
]

transaction_feature_names = [
    "has_transaction_data",
    "transaction_count",
    "total_paid",
    "avg_paid",
    "min_paid",
    "max_paid",
    "avg_plan_price",
    "avg_plan_days",
    "avg_discount",
    "auto_renew_rate",
    "last_auto_renew",
    "cancel_count",
    "cancel_rate",
    "last_is_cancel",
    "days_since_last_transaction",
]

log_feature_names = [
    "has_log_data",
    "active_days",
    "total_listening_secs",
    "avg_listening_secs_per_day",
    "median_listening_secs_per_day",
    "avg_unique_songs",
    "total_unique_song_events",
    "total_num_25",
    "total_num_50",
    "total_num_75",
    "total_num_985",
    "total_num_100",
    "completion_rate",
    "days_since_last_activity",
    "active_days_7d",
    "active_days_30d",
    "listening_secs_7d",
    "listening_secs_30d",
    "extreme_listening_day_count",
]

In [60]:
expected_features = (
    member_feature_names
    + transaction_feature_names
    + log_feature_names
)

missing_expected = (
    set(expected_features)
    - set(feature_columns)
)

unexpected_features = (
    set(feature_columns)
    - set(expected_features)
)

print("Missing expected:", missing_expected)
print("Unexpected:", unexpected_features)

Missing expected: set()
Unexpected: set()


## 13. Key Findings

### Dataset

- Final dataset có **970,960 users**.
- `msno` là unique.
- Final dataset có **42 columns**.
- Target không missing và chỉ chứa `0/1`.
- Invariant `1 user = 1 row` được đảm bảo.

### Source Coverage

- Member coverage: **88.67%**.
- Transaction coverage: **96.15%**.
- User log coverage: **77.71%**.

Coverage sau Feature Engineering khớp với QT2.

### Member Processing

- Invalid age được chuyển thành missing thay vì drop user.
- Missing gender được xử lý có chủ đích.
- Registration date được chuyển thành account tenure feature.

### Transaction Processing

- Transaction history được aggregate về user-level.
- Payment, auto-renew, cancellation và transaction recency features được tạo.
- `membership_expire_date` chưa được sử dụng trong V1 do anomaly được phát hiện ở QT2.

### User Activity Processing

- Daily logs được aggregate về user-level.
- Listening duration bất thường được xử lý trong feature pipeline thay vì thay đổi raw data.
- Activity frequency, duration, song-completion và recency features được tạo.
- Recent 7-day và 30-day activity features được tạo.

### Missing Values

Một số feature vẫn chứa missing có ý nghĩa:

- `age`: 584,245 users.
- `avg_paid`: 37,382 users.
- `avg_unique_songs`: 216,409 users.

QT3 không thực hiện global statistical imputation.

Imputation sẽ được fit trên training data trong QT4.

### Leakage

- Cutoff date: **2017-04-01**.
- Latest transaction used: **2017-03-31**.
- Latest activity used: **2017-03-31**.
- Không phát hiện time-based leakage.

### Testing

Toàn bộ QT3 validation tests đã pass:

```text
7 passed

# 14. Next Steps — QT4 Model Development

QT3 đã tạo dataset sẵn sàng cho modeling:

```text
data/processed/train_features.parquet

In [61]:
checks = {
    "processed_file_exists": TRAIN_FEATURES_PATH.exists(),
    "member_file_exists": MEMBER_FEATURES_PATH.exists(),
    "transaction_file_exists": TRANSACTION_FEATURES_PATH.exists(),
    "log_file_exists": USER_LOG_FEATURES_PATH.exists(),
    "one_row_per_user": (
        integrity.loc[0, "rows"]
        == integrity.loc[0, "users"]
    ),
    "target_complete": (
        integrity.loc[0, "missing_target"] == 0
    ),
    "target_valid": (
        integrity.loc[0, "invalid_target"] == 0
    ),
    "transaction_leakage_free": (
        transaction_leakage.loc[0, "invalid_users"] == 0
    ),
    "log_leakage_free": (
        log_leakage.loc[0, "invalid_users"] == 0
    ),
}

pd.Series(checks, name="passed")

processed_file_exists       True
member_file_exists          True
transaction_file_exists     True
log_file_exists             True
one_row_per_user            True
target_complete             True
target_valid                True
transaction_leakage_free    True
log_leakage_free            True
Name: passed, dtype: bool